In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch import Tensor
from torchvision import transforms
import h5py

import matplotlib.pyplot as plt

import gc
import math
import random

# for deleting old best models
import os
import shutil
import glob

from dotenv import load_dotenv


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

random.seed(42)

Device is cuda


In [2]:
load_dotenv("config.env")

# --- Dataset parameters ---
USE_DVS = int(os.getenv("USE_DVS"))
USE_APS = int(os.getenv("USE_APS"))
MODEL_DIMS = USE_DVS + USE_APS
if MODEL_DIMS == 0: raise Exception("MODEL_DIMS must be non-zero")

BINSIZE_MS = int(os.getenv("BINSIZE_MS"))
HEIGHT = int(os.getenv("HEIGHT"))
WIDTH = int(os.getenv("WIDTH"))
PRECISION = int(os.getenv("PRECISION"))
DTYPE = np.float16 if PRECISION == 16 else np.float32

APS_FRAME_NAME = f"aps_frame_{HEIGHT}x{WIDTH}"
DVS_FRAME_NAME = f"dvs_frame_{HEIGHT}x{WIDTH}"

# --- Location parameters ---
DATASET_DIR = os.getenv("OUT_DIR")
MODEL_SAVE_DIR = os.getenv("MODEL_SAVE_DIR")
# --- Preprocessing parameters ---
PERFORM_PRUNING = bool(int(os.getenv("PERFORM_PRUNING")))
# km/h for speed to be considered high enough
SPEED_THRESHOLD = float(os.getenv("SPEED_THRESHOLD"))
# absolute value for degree of steering to be considered high enough
STEERING_THRESHOLD = float(os.getenv("STEERING_THRESHOLD"))
STEERING_DISCARD_CHANCE = float(os.getenv("STEERING_DISCARD_CHANCE"))

# --- Training parameters ---
WARM_START= bool(int(os.getenv("WARM_START")))
CLEAN_OLD_MODELS = bool(int(os.getenv("CLEAN_OLD_MODELS")))
if WARM_START and CLEAN_OLD_MODELS: raise Exception("Can't have WARM_START and CLEAN_OLD_MODELS at the same time!")
USE_STATED= bool(int(os.getenv("USE_STATED")))
TRAINING_SPLIT = float(os.getenv("TRAINING_SPLIT"))
EPOCHS = int(os.getenv("EPOCHS"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY"))


In [3]:
# Randomly allocate a percentage of the data to the training set
TRAINING_DIR = f"{DATASET_DIR}/Training"
TEST_DIR = f"{DATASET_DIR}/Test"

# Try to create directory, shuffle the files back to the main directory if it already exists
try:
    os.mkdir(TRAINING_DIR)
except FileExistsError:
    for files in glob.glob(TRAINING_DIR + "/*.hdf5"):
        os.rename(files, f"{DATASET_DIR}/{os.path.basename(files)}")
    # delete everything that remains, a.k.a. pruned files
    for file in glob.glob(TRAINING_DIR + "/*"):
        if os.path.isdir(file):
            shutil.rmtree(file)
        else:
            os.remove(file)

try:
    os.mkdir(TEST_DIR)
except FileExistsError:
    for files in glob.glob(TEST_DIR + "/*"):
         os.rename(files, f"{DATASET_DIR}/{os.path.basename(files)}")


In [4]:
# Sort by name to make the randomness seedable
data_files = sorted(glob.glob(f"{DATASET_DIR}/*.hdf5"))

# Randomly shuffle the data files
random.shuffle(data_files)

split_idx = int(len(data_files) * TRAINING_SPLIT)
train_files = data_files[:split_idx]
test_files = data_files[split_idx:]

for file in train_files:
    os.rename(file, f"{TRAINING_DIR}/{os.path.basename(file)}")

for file in test_files:
    os.rename(file, f"{TEST_DIR}/{os.path.basename(file)}")


In [ ]:
# Only prune training data
if PERFORM_PRUNING:
    PRUNE_SPEED = bool(int(os.getenv("SPEED_PRUNING")))
    PRUNE_STEERING = bool(int(os.getenv("STEERING_PRUNING")))
    
    PRUNED_DATASET_PATH = TRAINING_DIR + "/Pruned-Recordings"
    os.mkdir(PRUNED_DATASET_PATH)
    
    training_files = glob.glob(f"{TRAINING_DIR}/*.hdf5")
    for file in training_files:
        print(f"Pruning file: {file}")
        discarded_batches = []
        with h5py.File(file, "r") as f:
            speeds = np.array(f["vehicle_speed"])
            steering_angles = np.array(f["steering_wheel_angle"])
            file_len = len(speeds)
            for i in range(0, len(speeds), BATCH_SIZE):
                if PRUNE_SPEED:
                    if np.average(speeds[i:i+BATCH_SIZE]) < SPEED_THRESHOLD:
                        discarded_batches.append(i)
                        continue
                if PRUNE_STEERING:
                    if np.average(np.absolute(steering_angles[i:i+BATCH_SIZE])) < STEERING_THRESHOLD and random.random() < STEERING_DISCARD_CHANCE:
                        discarded_batches.append(i)

            indices_to_remove = []
            for idx in discarded_batches:
                indices_to_remove.extend(range(idx, idx + BATCH_SIZE))
            indices_to_keep = np.setdiff1d(np.arange(file_len), indices_to_remove)
            renamed_file = f"{PRUNED_DATASET_PATH}/{os.path.splitext(os.path.basename(file))[0]}_pruned.hdf5"
            with h5py.File(renamed_file, "w") as new_f:
                for dset_name in f.keys():
                    if len(f[dset_name]) != file_len:
                        # print(f"Dataset {dset_name} has a different length than the rest of the data, skipping.")
                        # in practice this is just test_idxs, and train_idxs, which are not needed
                        continue
                    full_data = np.array(f[dset_name])
                    new_f.create_dataset(dset_name, data=full_data[indices_to_keep])
            
            print(f"Discarded {len(discarded_batches)} batches, as their average was below the thresholds, totalling {len(indices_to_remove)} frames.")
            print(f"{len(indices_to_keep)} frames remain, with average speed {np.average(speeds[indices_to_keep])} and average absolute steering {np.average(np.absolute(steering_angles[indices_to_keep]))}") 

    TRAINING_DIR = PRUNED_DATASET_PATH

In [6]:
class LazyDDD20Dataset(Dataset):
    '''
    adapted from https://stackoverflow.com/questions/74375033/pytorch-dataloader-for-multiple-files-with-sliding-window
    '''
    def __init__(self, data_folder, *, transform=None, target_transform=None, window_size=1, use_aps=True, use_dvs=True):
        if not (use_aps or use_dvs):
            raise ValueError("At least one of use_aps or use_dvs must be True.")
        self.transform = transform
        self.data_file_list = glob.glob(f"{data_folder}/*.hdf5")
        self.target_transform = target_transform
        self.window_size = window_size
        self.use_aps = use_aps
        self.use_dvs = use_dvs
        self.inv_apsdvs_max = 1 / 255
        self.apsdvs_min = 0
        
        self.total_windows, self.dataset_dict = self.create_dataset_dict()

    def find_num_windows(self, path_to_file):
        # Note the integer division here, drop_last is implied true which is OK.
        with h5py.File(path_to_file, "r") as f:
            return len(f["steering_wheel_angle"]) // self.window_size
    
    def create_dataset_dict(self):
        total_windows = 0
        overall_index = 0
        dataset_dict = {}
        for data_file in self.data_file_list:
            windows_of_current_file = self.find_num_windows(data_file)
            total_windows += windows_of_current_file

            # Show which overall index corresponds to which file and window index
            for i in range(windows_of_current_file):
                dataset_dict[overall_index] = {
                    "filename" : data_file,
                    "window_index" : i
                }
                overall_index += 1

        return total_windows, dataset_dict
    
    def read_file(self, filename, window_index):
        with h5py.File(filename, "r") as f:
            if self.use_dvs:
                    dvs_frame = np.array(f[DVS_FRAME_NAME][window_index:window_index + self.window_size])
            if self.use_aps:
                    aps_frame = np.array(f[APS_FRAME_NAME][window_index:window_index + self.window_size])
            if self.use_aps and self.use_dvs:
                # the frames are 1 x 224 x 224, we want 224 x 224 x 2 after concat
                combined_frames = np.concatenate((dvs_frame, aps_frame), axis=0)
            else:
                combined_frames = aps_frame if self.use_aps else dvs_frame
            
            combined_frames = combined_frames.transpose(1, 2, 0)
            steering_angles = np.array(f["steering_wheel_angle"][window_index:window_index + self.window_size])
        # print(f"Combined frames shape is {combined_frames.shape}.")
        return combined_frames, steering_angles
    
    # used to normalise the aps and dvs frames to the range [0, 1]
    def normalise(self, data):
        # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
        new_data = np.empty_like(data, dtype=DTYPE)
        new_data = data * self.inv_apsdvs_max
        return new_data

    def __len__(self):
        return self.total_windows

    def __getitem__(self, index):
        filename = self.dataset_dict[index]["filename"]
        window_index = self.dataset_dict[index]["window_index"]
        data, target = self.read_file(filename, window_index)
        data = self.normalise(data)
        
        if self.transform:
            data = self.transform(data)
        
        target = target.astype(DTYPE)
        if self.target_transform:
            target = self.target_transform(target)
        
        return data, target
        
    def print_dataset_dict(self):
        for key, value in self.dataset_dict.items():
            print(f"Index: {key}, Filename: {value['filename']}, Window Index: {value['window_index']}")

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

train_dataset = LazyDDD20Dataset(TRAINING_DIR, transform=transform, window_size=1, use_aps=USE_APS, use_dvs=USE_DVS)
test_dataset = LazyDDD20Dataset(TEST_DIR, transform=transform, window_size=1, use_aps=USE_APS, use_dvs=USE_DVS)

# load the data, can implement batching etc. here
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
print(f"Train loader has {len(train_loader)} batches, equalling {len(train_loader) * BATCH_SIZE} frames.\n-Test loader has {len(test_loader)} batches, equalling {len(test_loader) * BATCH_SIZE} frames.")

In [8]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1),
                        nn.BatchNorm2d(out_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
                        nn.BatchNorm2d(out_channels))
        self.downsample = downsample
        self.relu = nn.ReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out


In [9]:
class ResNetRNN(nn.Module):
    def __init__(self, block, layers, num_channels, num_classes = 1):
        super(ResNetRNN, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Sequential(
                        nn.Conv2d(num_channels, 64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm2d(64),
                        nn.ReLU())
        self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
        self.layer0 = self._make_layer(block, 64, layers[0], stride = 1)
        self.layer1 = self._make_layer(block, 128, layers[1], stride = 2)
        self.layer2 = self._make_layer(block, 256, layers[2], stride = 2)
        self.layer3 = self._make_layer(block, 512, layers[3], stride = 2)
        self.avgpool = nn.AvgPool2d(7, stride=1)
        self.lstm = nn.LSTM(input_size=512, hidden_size=128, num_layers=1)
        self.fc = nn.Linear(128, num_classes)
        self._init_weights()
    
    # initialise as per the DDD20 Paper.
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nin = m.in_channels if hasattr(m, 'in_channels') else m.in_features
                m.weight.data.normal_(0, math.sqrt(2.0 / nin))
                if m.bias is not None:
                    m.bias.data.zero_()  

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:

            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride),
                nn.BatchNorm2d(planes),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        # rint(f"Conv1 shape is {x.shape}")
        x = self.maxpool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        # print(f"shape of x after avgpool is {x.shape}")
        # x = x.view(x.size(0), -1)
        x = x.squeeze(-1)
        x = x.permute(0, 2, 1)
        '''
        Sequence length is the "batch size" as is used everywhere else.
        For each "batch", we must get the output of the LSTM for that batch.
        We will then feed those outputs as the respective outputs of the batches to the fc layer.
        '''
        lstm_out, _ = self.lstm(x)
        # output should be of size (sequence_size, batch_size, 512)
        # print(f"shape of lstm_out is {lstm_out.shape}")
        x = lstm_out.squeeze(1)
        x = self.fc(x)

        return x.squeeze(1)

In [10]:
class ResNetRNNStated(nn.Module):
    def __init__(self, block, layers, num_channels, num_classes=1):
        super(ResNetRNNStated, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Sequential(
            nn.Conv2d(num_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer0 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer1 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer2 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer3 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AvgPool2d(7, stride=1)
        self.lstm = nn.LSTM(input_size=512, hidden_size=128, num_layers=1)
        self.fc = nn.Linear(128, num_classes)
        self._init_weights()
        self.hidden = None

    # Initialize weights as per the DDD20 Paper.
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nin = m.in_channels if hasattr(m, 'in_channels') else m.in_features
                m.weight.data.normal_(0, math.sqrt(2.0 / nin))
                if m.bias is not None:
                    m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride),
                nn.BatchNorm2d(planes),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)

        # Detach hidden state to prevent backpropagating through the entire history
        if self.hidden is not None:
            self.hidden = tuple([h.detach() for h in self.hidden])

        x, self.hidden = self.lstm(x.unsqueeze(0), self.hidden)
        x = self.fc(x.squeeze(0))

        return x.squeeze(1)

    def reset_hidden(self):
        self.hidden = None

In [11]:
def setup_model(model_dims, use_stated):
    if use_stated:
        model = ResNetRNNStated(ResidualBlock, [3, 4, 6, 3], num_channels=model_dims).to(device)
    else:
        model = ResNetRNN(ResidualBlock, [3, 4, 6, 3], num_channels=model_dims).to(device)
    
    return model


In [12]:
def warm_start(model, optimizer):
        """
        Load the last model and optimizer state from the last saved model, returns starting epoch and best loss
        """
        last_model_file = glob.glob(f"{MODEL_SAVE_DIR}/last/*")[-1]
        load_dict = torch.load(last_model_file, map_location=device)
        model.load_state_dict(load_dict["model_state_dict"])
        optimizer.load_state_dict(load_dict["optimizer_state_dict"])
        epoch = load_dict["epoch"]
        best_test_loss = load_dict["best_test_loss"]
        return epoch, best_test_loss

In [13]:
model = setup_model(MODEL_DIMS, USE_STATED)

starting_epoch = 0
best_test_loss = np.inf

#Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

if WARM_START:
    starting_epoch, best_test_loss = warm_start(model, optimizer)

In [14]:
def clean_models(directory):
    '''
    Delete the models in a given directory
    '''
    for model_file in glob.glob(directory + "/*.pth"):
        os.remove(model_file)

In [15]:
def save_model(directory, model, optimizer, epoch, loss, best_test_loss, *, clean_old=False):
    '''
    Saves the model in the given location.
    If clean_old, cleans the old model files in the directory
    '''
    state = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'loss': loss,
                'best_test_loss': best_test_loss
    }
    
    if clean_old:
        clean_models(directory)
    
    torch.save(state, f"{directory}/e{epoch + 1}_L{loss:.2f}.pth")

In [ ]:
if CLEAN_OLD_MODELS:
  clean_models(MODEL_SAVE_DIR)

train_losses = []
test_losses = []

scaler = torch.amp.GradScaler("cuda")
for epoch in range(starting_epoch, EPOCHS):
    epoch_train_loss = 0
    for combined_images, steering_angle in train_loader:
      optimizer.zero_grad()
      #Move tensors to the configured device
      combined_images = combined_images.to(device)
      steering_angle = steering_angle.to(device)
      #Forward pass
      with torch.autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(combined_images)
        # print(f"o {outputs.dtype} og {steering_angle.dtype}")
        loss = criterion(outputs, steering_angle.squeeze(1))
      #Backward and optimize
      scaler.scale(loss).backward()
      scaler.step(optimizer)
      scaler.update()
      epoch_train_loss += loss.item()
      del combined_images, steering_angle, outputs
      torch.cuda.empty_cache()
      gc.collect()

    # Average training loss for the epoch
    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)

    #Validation
    epoch_test_loss = 0
    with torch.no_grad():
          for combined_images, steering_angle in test_loader:
              combined_images = combined_images.to(device)
              steering_angle = steering_angle.to(device)
              with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(combined_images)
                loss = criterion(outputs, steering_angle.squeeze(1))
              epoch_test_loss += loss.item()
              del combined_images, steering_angle, outputs
          epoch_test_loss /= len(test_loader)
          
          # Save the best
          if (epoch_test_loss < best_test_loss):
              best_test_loss = epoch_test_loss
              save_model(f"{MODEL_SAVE_DIR}/best", model, optimizer, epoch, epoch_test_loss, best_test_loss, clean_old=True)

          # Save every 20 epochs
          if not ((epoch + 1) % 20):
              save_model(MODEL_SAVE_DIR, model, optimizer, epoch, epoch_test_loss, best_test_loss)
          # Save the last model
          save_model(f"{MODEL_SAVE_DIR}/last", model, optimizer, epoch, epoch_test_loss, best_test_loss, clean_old=True)

          # Report on the loss
          test_losses.append(epoch_test_loss)
          print(f"Epoch [{epoch + 1}/{EPOCHS}], Train Loss: {epoch_train_loss: .4f} Test Loss: {epoch_test_loss: .4f}")
            
# Plotting the training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss')
plt.plot(range(1, EPOCHS + 1), test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (Degrees of Steering)')
plt.title('Training and Test Losses')
plt.legend()
plt.show()

In [ ]:
# Function to generate predictions in batches
def generate_predictions(model, data_X, data_y, batch_size, device, transform):
    model.eval()
    predictions = []
    with torch.no_grad():
        for i in range(0, len(data_X) - len(data_X) % batch_size, batch_size):
            batch_X = data_X[i:i+batch_size]
            batch_y = data_y[i:i+batch_size]
            transformed_batch_X = torch.stack([transform(img) for img in batch_X])
            batch_X_tensor = transformed_batch_X.to(device)
            batch_y_tensor = Tensor(batch_y).to(device)
            outputs = model(batch_X_tensor)
            predictions.extend(outputs.detach().cpu().numpy())
            del batch_X_tensor, batch_y_tensor, outputs
            torch.cuda.empty_cache()
            gc.collect()
    return predictions


In [ ]:
def generate_predictions(model, file_path, batch_size=1024):
    model.eval()

    with h5py.File(file_path, "r") as f:
        dvs_frame = np.array(f[DVS_FRAME_NAME])
        data_len = dvs_frame.shape[0]
        aps_frame = np.array(f[APS_FRAME_NAME])
        steering_angles = np.array(f["steering_wheel_angle"])
        if USE_APS and USE_DVS:
            # the frames are 1 x 224 x 224, we want 224 x 224 x 2 after concat
            combined_frames = np.concatenate((dvs_frame, aps_frame), axis=0)
        else:
            combined_frames = aps_frame if USE_APS else dvs_frame

        combined_frames = combined_frames.transpose(1, 2, 0)
        transform(combined_frames)

        preds, gts = [], []
        for start in range(0, data_len, batch_size):
            end = min(start + batch_size, data_len)
            batch_data = combined_frames[start:end]

            with torch.no_grad():
                batch_out = model(batch_data).cpu().numpy()

            preds.append(batch_out)
            gts.append(steering_angles[start:end])

        preds = np.concatenate(preds, axis=0)
        gts = np.concatenate(gts, axis=0)

        return preds, gts

In [ ]:

# Iterate through every file in the testdir
test_files = glob.glob(f"{TEST_DIR}/*.hdf5")
plt.figure(figsize=(15,7.5 * len(test_files)))

for i, f in enumerate(test_files):
    preds, gts = generate_predictions(model, f)
    plt.subplot(len(test_files), 1, i)
    plt.plot(gts, label="Ground truth")
    plt.plot(preds, label="Model Predictions")
    plt.title(f)
    plt.xlabel("Data Point")
    plt.ylabel("Steering Angle")
    plt.legend()

plt.tight_layout()
plt.show()